In [1]:
# ============================================
# 04_evaluation_ablation.ipynb
# Purpose: LLM-as-a-Judge evaluation and ablation study
# ============================================

# Cell 1: Import libraries
import json
import requests
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import time
from dotenv import load_dotenv
import os
from typing import List, Dict, Tuple
import pandas as pd

load_dotenv()

print("Libraries loaded!")


d:\IBA\Sem-6\NLP with Deep Learning\Assignment 3\pakistan_history-rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded!


In [2]:
# ============================================
# Cell 2: Setup LLM using Groq API (UPDATED WITH CURRENT MODELS)
# Meets assignment: LLM Generation via HF Inference API compatible
# ============================================

import requests
import time
import re
import os
from dotenv import load_dotenv

load_dotenv()

# Get Groq API key from .env file
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    print("❌ ERROR: GROQ_API_KEY not found in .env file!")
    print("\n📋 To get a free Groq API key:")
    print("   1. Go to https://console.groq.com")
    print("   2. Sign up (free, takes 1 minute)")
    print("   3. Go to 'API Keys' section")
    print("   4. Click 'Create API Key'")
    print("   5. Copy the key")
    print("   6. Add to your .env file: GROQ_API_KEY=your_key_here")
    raise ValueError("Missing Groq API key")

# ============================================
# CURRENT AVAILABLE MODELS ON GROQ (as of 2026)
# ============================================
# Options:
# - "llama-3.3-70b-versatile" (Most powerful, recommended)
# - "llama-3.1-8b-instant" (Faster, good for evaluation)
# - "gemma2-9b-it" (Google's model)
# - "mixtral-8x7b-32768" (DECOMMISSIONED - don't use)

# Using Llama 3.3 70B (powerful, meets assignment requirements)
SELECTED_MODEL = "llama-3.3-70b-versatile"  # Alternative: "llama-3.1-8b-instant"

# Groq API configuration
GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {GROQ_API_KEY}",
    "Content-Type": "application/json"
}

print("="*60)
print("LLM CONFIGURATION")
print("="*60)
print(f"✅ Using Groq API (HF Inference API compatible)")
print(f"   Model: {SELECTED_MODEL}")
print(f"   Provider: Groq Cloud")
print(f"   Purpose: LLM-as-a-Judge for Faithfulness & Relevancy")

def query_llm(prompt: str, max_retries=3) -> str:
    """
    Query Groq API for LLM response
    This replaces the Hugging Face Inference API
    """
    payload = {
        "model": SELECTED_MODEL,
        "messages": [
            {
                "role": "system", 
                "content": "You are a helpful assistant for evaluating RAG systems. Respond accurately and concisely with the requested format."
            },
            {
                "role": "user", 
                "content": prompt
            }
        ],
        "temperature": 0.1,  # Low temperature for consistent evaluation
        "max_tokens": 500,
        "top_p": 0.95
    }
    
    for attempt in range(max_retries):
        try:
            response = requests.post(GROQ_API_URL, headers=headers, json=payload, timeout=30)
            
            if response.status_code == 200:
                result = response.json()
                return result['choices'][0]['message']['content']
                
            elif response.status_code == 400:
                error_msg = response.json().get('error', {}).get('message', 'Unknown error')
                print(f"⚠ Error 400: {error_msg[:100]}")
                if "decommissioned" in error_msg.lower():
                    print("   Model is deprecated. Please update SELECTED_MODEL")
                time.sleep(2)
                
            elif response.status_code == 401:
                print(f"❌ Authentication error: Invalid Groq API key")
                return "ERROR: Invalid API key"
                
            elif response.status_code == 429:
                wait_time = 3
                print(f"⚠ Rate limit, waiting {wait_time}s... (attempt {attempt+1}/{max_retries})")
                time.sleep(wait_time)
                
            elif response.status_code == 503:
                print(f"⚠ Service busy, waiting 5s... (attempt {attempt+1}/{max_retries})")
                time.sleep(5)
                
            else:
                print(f"⚠ Error {response.status_code}: {response.text[:100]}")
                time.sleep(2)
                
        except requests.exceptions.Timeout:
            print(f"⚠ Timeout, retrying... (attempt {attempt+1}/{max_retries})")
            time.sleep(2)
        except Exception as e:
            print(f"⚠ Error: {str(e)[:100]}, retrying...")
            time.sleep(2)
    
    return "ERROR: Could not get response from LLM after multiple retries"

# Test the connection with the new model
print("\n" + "="*60)
print("TESTING LLM CONNECTION")
print("="*60)

test_prompt = "Respond with exactly one word: OK"
try:
    response = query_llm(test_prompt)
    
    if "ERROR" in response:
        print(f"❌ LLM connection failed: {response}")
        print("\nTroubleshooting:")
        print("1. Check your API key is correct in .env file")
        print("2. Make sure you have credits (Groq gives free credits on signup)")
        print("3. Check your internet connection")
        print("\n🔄 Trying alternative model (llama-3.1-8b-instant)...")
        
        # Try alternative model
        SELECTED_MODEL = "llama-3.1-8b-instant"
        response = query_llm(test_prompt)
        
        if "ERROR" not in response:
            print(f"✅ Alternative model working! Using {SELECTED_MODEL}")
    else:
        print(f"✅ LLM connection successful!")
        print(f"   Response: {response}")
        
except Exception as e:
    print(f"❌ Connection error: {e}")

print("\n" + "="*60)
print("✅ LLM EVALUATION SETUP COMPLETE!")
print("="*60)
print(f"📊 LLM Provider: Groq (replaces HF Inference API)")
print(f"📊 Model: {SELECTED_MODEL}")
print(f"📊 Features: Faithfulness & Relevancy LLM-as-a-Judge")

LLM CONFIGURATION
✅ Using Groq API (HF Inference API compatible)
   Model: llama-3.3-70b-versatile
   Provider: Groq Cloud
   Purpose: LLM-as-a-Judge for Faithfulness & Relevancy

TESTING LLM CONNECTION
✅ LLM connection successful!
   Response: OK

✅ LLM EVALUATION SETUP COMPLETE!
📊 LLM Provider: Groq (replaces HF Inference API)
📊 Model: llama-3.3-70b-versatile
📊 Features: Faithfulness & Relevancy LLM-as-a-Judge


In [3]:
# ============================================
# Cell 3: Define test queries (10-20 as required)
# ============================================

test_queries = [
    # Political history
    "When did Pakistan gain independence from British rule?",
    "Who was the first Governor-General of Pakistan?",
    "What was the Objective Resolution of 1949?",
    
    # Military history
    "What were the main causes of the Indo-Pakistani War of 1965?",
    "How did Bangladesh separate from Pakistan in 1971?",
    "What was the Kargil War and when did it occur?",
    
    # Ancient history
    "What is the significance of the Indus Valley Civilization in Pakistan's history?",
    "Which ancient empires ruled the region that is now Pakistan?",
    
    # Political figures
    "What were Muhammad Ali Jinnah's key contributions to the Pakistan Movement?",
    "What reforms did Zulfikar Ali Bhutto implement during his premiership?",
    
    # Constitutional history
    "When was the first Constitution of Pakistan adopted?",
    "What is the significance of the Lahore Resolution of 1940?",
    
    # Cultural history
    "How did the Khilafat Movement influence the Pakistan Movement?",
    "What was the role of Allama Iqbal in the creation of Pakistan?",
    
    # Post-independence
    "What were the economic policies of General Muhammad Zia-ul-Haq's regime?",
    "How did Pakistan become a nuclear power?"
]

print(f"📋 Defined {len(test_queries)} test queries for evaluation")

📋 Defined 16 test queries for evaluation


In [4]:
# ============================================
# Cell 4: Function to generate answers using retrieved context (FIXED VERSION)
# ============================================

import pickle
import sys
import os
import numpy as np
import re
from collections import defaultdict
from dotenv import load_dotenv

load_dotenv()

print("Loading components...")

# Initialize embedding model
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print("✅ Embedding model loaded")

# Initialize cross-encoder
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("✅ Cross-encoder loaded")

# Connect to Pinecone
from pinecone import Pinecone
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("pakistan-history-rag")
print("✅ Pinecone connected")

# ============================================
# Recreate BM25 retriever from chunks (instead of loading)
# ============================================

from rank_bm25 import BM25Okapi

def tokenize_text(text):
    """Simple tokenizer - lowercase and split on non-alphanumeric"""
    return re.findall(r'\w+', text.lower())

# Load the chunks from Phase 1 (since we have them)
print("\nLoading chunks to build BM25 index...")

with open('chunks_recursive.json', 'r') as f:
    import json
    all_chunks = json.load(f)

print(f"✅ Loaded {len(all_chunks)} chunks")

# Build BM25 index
print("Building BM25 index...")
tokenized_chunks = [tokenize_text(chunk['text']) for chunk in all_chunks]
bm25 = BM25Okapi(tokenized_chunks)
print("✅ BM25 index built")

# Create BM25 retriever class
class BM25Retriever:
    def __init__(self, bm25_obj, chunks, tokenizer_func):
        self.bm25 = bm25_obj
        self.chunks = chunks
        self.tokenize = tokenizer_func
    
    def search(self, query, top_k=10):
        tokenized_query = self.tokenize(query)
        scores = self.bm25.get_scores(tokenized_query)
        top_indices = np.argsort(scores)[::-1][:top_k]
        
        results = []
        for idx in top_indices:
            if scores[idx] > 0:
                results.append({
                    'id': self.chunks[idx].get('id', f"chunk_{idx}"),
                    'text': self.chunks[idx]['text'],
                    'score': float(scores[idx]),
                    'metadata': self.chunks[idx]
                })
        return results

bm25_retriever = BM25Retriever(bm25, all_chunks, tokenize_text)
print("✅ BM25 retriever ready")

# ============================================
# Define retrieval functions
# ============================================

def reciprocal_rank_fusion(semantic_results, bm25_results, k=60):
    """Combine search results using RRF"""
    fusion_scores = defaultdict(float)
    result_data = {}
    
    for rank, result in enumerate(semantic_results, start=1):
        result_id = result['id']
        fusion_scores[result_id] += 1 / (k + rank)
        result_data[result_id] = result
    
    for rank, result in enumerate(bm25_results, start=1):
        result_id = result['id']
        fusion_scores[result_id] += 1 / (k + rank)
        if result_id not in result_data:
            result_data[result_id] = result
    
    sorted_results = sorted(fusion_scores.items(), key=lambda x: x[1], reverse=True)
    final_results = []
    for result_id, score in sorted_results:
        result = result_data[result_id].copy()
        result['fusion_score'] = score
        final_results.append(result)
    
    return final_results

def hybrid_search(query, top_k=10):
    """Hybrid search combining semantic and BM25"""
    # Semantic search
    query_embedding = embedding_model.encode(query)
    semantic_results_raw = index.query(
        vector=query_embedding.tolist(),
        top_k=top_k*2,
        include_metadata=True
    )
    
    semantic_results = []
    for match in semantic_results_raw['matches']:
        semantic_results.append({
            'id': match['id'],
            'text': match['metadata'].get('text', ''),
            'score': match['score'],
            'metadata': match['metadata']
        })
    
    # BM25 search
    bm25_results = bm25_retriever.search(query, top_k=top_k*2)
    
    # Fuse results
    fused_results = reciprocal_rank_fusion(semantic_results, bm25_results)
    
    return fused_results[:top_k]

def rerank_results(query, results):
    """Re-rank results using cross-encoder"""
    if not results:
        return results
    
    # Get text from results (handle both formats)
    def get_text(r):
        if 'text' in r:
            return r['text']
        elif 'metadata' in r and 'text' in r['metadata']:
            return r['metadata']['text']
        else:
            return ""
    
    pairs = [(query, get_text(result)) for result in results]
    scores = cross_encoder.predict(pairs)
    
    for i, result in enumerate(results):
        result['rerank_score'] = float(scores[i])
    
    return sorted(results, key=lambda x: x['rerank_score'], reverse=True)

def get_answer(query, use_rerank=True, top_k=5):
    """Get answer for a query using RAG"""
    # Retrieve
    results = hybrid_search(query, top_k=20)
    if use_rerank:
        results = rerank_results(query, results)
    
    final_chunks = results[:top_k]
    context = "\n\n---\n\n".join([r['text'] for r in final_chunks])
    
    # Generate answer using LLM
    prompt = f"""You are a helpful assistant that answers questions about Pakistan's history. 
Use ONLY the following context to answer the question. If the context doesn't contain the answer, say "I don't have enough information to answer that."

Context:
{context}

Question: {query}

Answer (be concise and factual):"""
    
    answer = query_llm(prompt)
    
    return {
        'query': query,
        'answer': answer,
        'retrieved_chunks': final_chunks,
        'context': context
    }

print("\n" + "="*60)
print("✅ ALL COMPONENTS READY!")
print("="*60)
print(f"📊 BM25 index size: {len(all_chunks)} chunks")
print(f"📊 Embedding model: all-MiniLM-L6-v2")
print(f"📊 Cross-encoder: ms-marco-MiniLM-L-6-v2")
print(f"📊 Pinecone index: pakistan-history-rag")

Loading components...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7652.21it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 12407.78it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Cross-encoder loaded
✅ Pinecone connected

Loading chunks to build BM25 index...
✅ Loaded 12228 chunks
Building BM25 index...
✅ BM25 index built
✅ BM25 retriever ready

✅ ALL COMPONENTS READY!
📊 BM25 index size: 12228 chunks
📊 Embedding model: all-MiniLM-L6-v2
📊 Cross-encoder: ms-marco-MiniLM-L-6-v2
📊 Pinecone index: pakistan-history-rag


In [5]:
# ============================================
# Cell 5: Implement Faithfulness Evaluation
# ============================================

def evaluate_faithfulness(answer: str, context: str) -> Dict:
    """
    Evaluate faithfulness using LLM-as-a-Judge
    This checks if the answer is supported by the context
    """
    prompt = f"""You are an expert evaluator for RAG systems. Your task is to check if the answer is FAITHFUL to the provided context.

Faithfulness means every claim in the answer is directly supported by or can be inferred from the context.

Context:
{context}

Answer:
{answer}

Instructions:
1. Extract ALL factual claims from the answer
2. For each claim, verify if it's supported by the context
3. Calculate the faithfulness score = (number of supported claims / total claims)

Respond in this exact format:

CLAIMS:
[Claim 1]
[Claim 2]
...

VERIFICATION:
Claim 1: [SUPPORTED/NOT SUPPORTED] - [brief explanation]
Claim 2: [SUPPORTED/NOT SUPPORTED] - [brief explanation]
...

SCORE: [X.XX] (as a decimal between 0 and 1)

SUMMARY: [One sentence summary]"""

    response = query_llm(prompt)
    
    # Parse the response
    score = 0.0
    try:
        # Find the score line
        for line in response.split('\n'):
            if 'SCORE:' in line:
                score_str = line.split('SCORE:')[1].strip()
                score = float(score_str)
                break
    except:
        print(f"Could not parse score from: {response[:200]}")
        score = 0.5  # Default if parsing fails
    
    return {
        'faithfulness_score': score,
        'detailed_evaluation': response
    }

# Test faithfulness evaluation
test_answer = "Pakistan gained independence in 1947 from British rule."
test_context = "The Indian Independence Act 1947 led to the creation of Pakistan on August 14, 1947."
result = evaluate_faithfulness(test_answer, test_context)
print(f"Faithfulness Score: {result['faithfulness_score']}")
print(f"Evaluation:\n{result['detailed_evaluation'][:300]}...")

Faithfulness Score: 1.0
Evaluation:
CLAIMS:
1. Pakistan gained independence in 1947
2. Pakistan gained independence from British rule

VERIFICATION:
Claim 1: SUPPORTED - The context mentions the creation of Pakistan on August 14, 1947, which implies gaining independence.
Claim 2: SUPPORTED - The context implies British rule by mention...


In [6]:
# ============================================
# Cell 6: Implement Relevancy Evaluation
# ============================================

def evaluate_relevancy(query: str, answer: str) -> Dict:
    """
    Evaluate relevancy by generating alternate questions and computing similarity
    """
    # Generate 3 alternate questions from the answer
    prompt = f"""Based on the following answer, generate 3 different questions that this answer could be responding to.
The questions should be natural and cover different aspects of the answer.

Answer: {answer}

Generate 3 questions, one per line, starting with Q1:, Q2:, Q3:"""

    response = query_llm(prompt)
    
    # Parse generated questions
    generated_questions = []
    for line in response.split('\n'):
        if line.startswith('Q1:') or line.startswith('Q2:') or line.startswith('Q3:'):
            q = line.split(':', 1)[1].strip()
            generated_questions.append(q)
    
    # If parsing failed, use fallback
    if len(generated_questions) != 3:
        generated_questions = [
            f"What information is provided about {answer[:50]}...?",
            f"Can you elaborate on {answer[:50]}...?",
            f"What are the key points in: {answer[:50]}...?"
        ]
    
    # Compute cosine similarity between original query and generated questions
    # First, embed all texts
    all_texts = [query] + generated_questions
    embeddings = embedding_model.encode(all_texts)
    
    query_embedding = embeddings[0]
    question_embeddings = embeddings[1:]
    
    similarities = []
    for q_emb in question_embeddings:
        sim = cosine_similarity([query_embedding], [q_emb])[0][0]
        similarities.append(float(sim))
    
    avg_relevancy = np.mean(similarities)
    
    return {
        'relevancy_score': avg_relevancy,
        'individual_scores': similarities,
        'generated_questions': generated_questions
    }

# Test relevancy evaluation
result = evaluate_relevancy("When did Pakistan get independence?", test_answer)
print(f"Relevancy Score: {result['relevancy_score']:.3f}")
print(f"Generated questions: {result['generated_questions']}")

Relevancy Score: 0.823
Generated questions: ['When did Pakistan become an independent country?', 'What significant event occurred in Pakistan in the year 1947?', 'From which colonial power did Pakistan gain its independence?']


In [7]:
# ============================================
# Cell 7: Run complete evaluation on all test queries
# ============================================

def evaluate_system_for_queries(queries, use_rerank=True):
    """
    Evaluate the system on a set of queries
    """
    results = []
    
    for i, query in enumerate(queries):
        print(f"\n📝 Evaluating query {i+1}/{len(queries)}: {query[:50]}...")
        
        # Get answer
        answer_data = get_answer(query, use_rerank=use_rerank)
        
        # Evaluate faithfulness
        faith_eval = evaluate_faithfulness(answer_data['answer'], answer_data['context'])
        
        # Evaluate relevancy
        rel_eval = evaluate_relevancy(query, answer_data['answer'])
        
        results.append({
            'query': query,
            'answer': answer_data['answer'],
            'faithfulness_score': faith_eval['faithfulness_score'],
            'relevancy_score': rel_eval['relevancy_score'],
            'faithfulness_detail': faith_eval['detailed_evaluation'][:500],
            'generated_questions': rel_eval['generated_questions'],
            'retrieved_chunks_count': len(answer_data['retrieved_chunks'])
        })
        
        # Small delay to avoid rate limits
        time.sleep(1)
    
    return results

# Run evaluation for different configurations (Ablation Study)
print("="*60)
print("RUNNING ABLATION STUDY")
print("="*60)

# Configuration 1: Fixed chunking + Semantic only (baseline)
print("\n⚙️ Config 1: Fixed Chunking + Semantic Only")
# Note: For semantic only, we modify get_answer temporarily
def get_answer_semantic_only(query, top_k=5):
    query_embedding = embedding_model.encode(query)
    results = index.query(
        vector=query_embedding.tolist(),
        top_k=top_k,
        include_metadata=True
    )
    context = "\n\n---\n\n".join([m['metadata']['text'] for m in results['matches']])
    prompt = f"Answer based on context: {context}\n\nQuestion: {query}\nAnswer:"
    answer = query_llm(prompt)
    return {'answer': answer, 'context': context}

# We'll run evaluation on a subset for the ablation study (first 10 queries)
ablation_queries = test_queries[:10]

# Config 1: Semantic only (using fixed chunks - we need to ensure we use fixed chunks)
# For simplicity, we'll use the current index which has both strategies
# In a real study, you'd create separate indexes

print("\n📊 Running ablation experiments (this will take a while)...")
print("Note: For complete ablation, you'd need separate indexes for each chunking strategy")
print("We'll demonstrate the evaluation methodology with our current system")

# For demonstration, we'll run Config 2 and 3
config_results = {}

# Config 2: Hybrid + No Rerank
print("\n🔧 Config 2: Hybrid Search + No Re-ranking")
config_results['Hybrid_NoRerank'] = evaluate_system_for_queries(ablation_queries, use_rerank=False)

# Config 3: Hybrid + Re-ranking (Best model)
print("\n🔧 Config 3: Hybrid Search + Re-ranking")
config_results['Hybrid_WithRerank'] = evaluate_system_for_queries(ablation_queries, use_rerank=True)

RUNNING ABLATION STUDY

⚙️ Config 1: Fixed Chunking + Semantic Only

📊 Running ablation experiments (this will take a while)...
Note: For complete ablation, you'd need separate indexes for each chunking strategy
We'll demonstrate the evaluation methodology with our current system

🔧 Config 2: Hybrid Search + No Re-ranking

📝 Evaluating query 1/10: When did Pakistan gain independence from British r...

📝 Evaluating query 2/10: Who was the first Governor-General of Pakistan?...

📝 Evaluating query 3/10: What was the Objective Resolution of 1949?...

📝 Evaluating query 4/10: What were the main causes of the Indo-Pakistani Wa...

📝 Evaluating query 5/10: How did Bangladesh separate from Pakistan in 1971?...

📝 Evaluating query 6/10: What was the Kargil War and when did it occur?...

📝 Evaluating query 7/10: What is the significance of the Indus Valley Civil...

📝 Evaluating query 8/10: Which ancient empires ruled the region that is now...

📝 Evaluating query 9/10: What were Muhammad Ali Ji

In [8]:
# ============================================
# Cell 8: Create Ablation Study Table
# ============================================

# Calculate averages for each configuration
ablation_table = []
for config_name, results in config_results.items():
    avg_faithfulness = np.mean([r['faithfulness_score'] for r in results])
    avg_relevancy = np.mean([r['relevancy_score'] for r in results])
    
    ablation_table.append({
        'Configuration': config_name,
        'Avg Faithfulness': f"{avg_faithfulness:.2f}",
        'Avg Relevancy': f"{avg_relevancy:.2f}",
        'Avg Chunks Retrieved': f"{np.mean([r['retrieved_chunks_count'] for r in results]):.1f}"
    })

df_ablation = pd.DataFrame(ablation_table)
print("\n" + "="*60)
print("ABLATION STUDY RESULTS TABLE")
print("="*60)
print(df_ablation.to_string(index=False))

# Save results
df_ablation.to_csv('ablation_study_results.csv', index=False)


ABLATION STUDY RESULTS TABLE
    Configuration Avg Faithfulness Avg Relevancy Avg Chunks Retrieved
  Hybrid_NoRerank             0.77          0.49                  5.0
Hybrid_WithRerank             1.00          0.64                  5.0


In [9]:
# ============================================
# Cell 10: Semantic-Only Evaluation
# Runs the same 10-query evaluation using ONLY dense semantic search
# (no BM25, no re-ranking) to complete the ablation study table.
# Results are merged with the existing ablation CSV.
# ============================================

import time
import numpy as np
import pandas as pd

eval_queries = test_queries[:10]

print("=" * 60)
print("SEMANTIC-ONLY EVALUATION (Ablation Row)")
print("=" * 60)
print(f"Running on {len(eval_queries)} queries...\n")

def get_answer_semantic_only(query, top_k=5):
    """Retrieve using dense semantic search only, then generate answer."""
    t0 = time.perf_counter()
    query_embedding = embedding_model.encode(query)
    results_raw = index.query(
        vector=query_embedding.tolist(),
        top_k=top_k,
        include_metadata=True
    )
    retrieval_time = time.perf_counter() - t0

    chunks = []
    for match in results_raw["matches"]:
        text = match.get("metadata", {}).get("text", "")
        chunks.append({"id": match["id"], "text": text,
                       "score": match["score"],
                       "metadata": match.get("metadata", {})})

    context = "\n\n---\n\n".join([c["text"] for c in chunks[:top_k]])

    generation_prompt = f"""You are a helpful assistant answering questions about Pakistan's history.
Use ONLY the following context. If the context doesn't contain the answer, say "I don't have enough information."

Context:
{context[:3000]}

Question: {query}

Answer (concise and factual):"""

    answer = query_llm(generation_prompt)
    return {"answer": answer, "context": context,
            "retrieved_chunks": chunks, "retrieval_time": retrieval_time}


semantic_results = []

for i, query in enumerate(eval_queries):
    print(f"  [{i+1:2d}/{len(eval_queries)}] {query[:60]}...")

    result     = get_answer_semantic_only(query)
    faith_eval = evaluate_faithfulness(result["answer"], result["context"])
    rel_eval   = evaluate_relevancy(query, result["answer"])

    semantic_results.append({
        "query":              query,
        "answer":             result["answer"],
        "faithfulness_score": faith_eval["faithfulness_score"],
        "relevancy_score":    rel_eval["relevancy_score"],
        "retrieved_chunks_count": len(result["retrieved_chunks"]),
    })

    print(f"         faith={faith_eval['faithfulness_score']:.2f}  "
          f"rel={rel_eval['relevancy_score']:.2f}")

avg_faith = np.mean([r["faithfulness_score"] for r in semantic_results])
avg_rel   = np.mean([r["relevancy_score"]    for r in semantic_results])
avg_chunks = np.mean([r["retrieved_chunks_count"] for r in semantic_results])

print()
print("=" * 60)
print(f"Semantic Only  |  Faithfulness: {avg_faith:.2f}  |  Relevancy: {avg_rel:.2f}")
print("=" * 60)

# ── Merge with existing ablation CSV ─────────────────────────────────────
new_row = {
    "Configuration":       "Semantic_Only",
    "Avg Faithfulness":    round(avg_faith, 2),
    "Avg Relevancy":       round(avg_rel,   2),
    "Avg Chunks Retrieved": round(avg_chunks, 1),
}

try:
    df_existing = pd.read_csv("ablation_study_results.csv")
except FileNotFoundError:
    df_existing = pd.DataFrame(columns=new_row.keys())

# Remove old Semantic_Only row if present, then prepend new one
df_existing = df_existing[df_existing["Configuration"] != "Semantic_Only"]
df_updated  = pd.concat(
    [pd.DataFrame([new_row]), df_existing], ignore_index=True
)

df_updated.to_csv("ablation_study_results.csv", index=False)

print()
print("UPDATED ABLATION TABLE")
print("=" * 60)
print(df_updated.to_string(index=False))
print()
print("✅ ablation_study_results.csv updated with Semantic_Only row")


SEMANTIC-ONLY EVALUATION (Ablation Row)
Running on 10 queries...

  [ 1/10] When did Pakistan gain independence from British rule?...
⚠ Rate limit, waiting 3s... (attempt 1/3)
⚠ Rate limit, waiting 3s... (attempt 1/3)
         faith=1.00  rel=0.63
  [ 2/10] Who was the first Governor-General of Pakistan?...
⚠ Rate limit, waiting 3s... (attempt 1/3)
         faith=1.00  rel=0.48
  [ 3/10] What was the Objective Resolution of 1949?...
⚠ Rate limit, waiting 3s... (attempt 1/3)
⚠ Rate limit, waiting 3s... (attempt 1/3)
         faith=1.00  rel=0.49
  [ 4/10] What were the main causes of the Indo-Pakistani War of 1965?...
⚠ Rate limit, waiting 3s... (attempt 1/3)
         faith=0.00  rel=0.10
  [ 5/10] How did Bangladesh separate from Pakistan in 1971?...
⚠ Rate limit, waiting 3s... (attempt 1/3)
⚠ Rate limit, waiting 3s... (attempt 1/3)
         faith=1.00  rel=0.78
  [ 6/10] What was the Kargil War and when did it occur?...
⚠ Rate limit, waiting 3s... (attempt 1/3)
⚠ Rate limit, waiting 3

In [10]:
# ============================================
# Cell 9: Generate Example Outputs for Report (3 queries as required)
# ============================================

print("\n" + "="*60)
print("EXAMPLE OUTPUTS FOR REPORT (3 Queries)")
print("="*60)

# Pick 3 diverse queries for detailed reporting
example_queries = test_queries[:3]

for query in example_queries:
    print(f"\n{'='*50}")
    print(f"QUERY: {query}")
    print('='*50)
    
    # Get answer with best configuration
    result = get_answer(query, use_rerank=True)
    
    print(f"\n📝 ANSWER:\n{result['answer']}")
    
    print(f"\n📚 RETRIEVED CONTEXT (first 300 chars of top chunk):")
    if result['retrieved_chunks']:
        print(result['retrieved_chunks'][0]['text'][:300] + "...")
    
    # Evaluate
    faith = evaluate_faithfulness(result['answer'], result['context'])
    relev = evaluate_relevancy(query, result['answer'])
    
    print(f"\n📊 EVALUATION SCORES:")
    print(f"  Faithfulness: {faith['faithfulness_score']:.2f}")
    print(f"  Relevancy: {relev['relevancy_score']:.2f}")
    
    print(f"\n🔍 FAITHFULNESS DETAIL (first 500 chars):")
    print(faith['detailed_evaluation'][:500])
    
    print(f"\n💡 GENERATED QUESTIONS FOR RELEVANCY:")
    for i, q in enumerate(relev['generated_questions'], 1):
        print(f"  {i}. {q}")
    
    print("\n" + "-"*50)

# Save all results for report
all_eval_results = {
    'ablation_table': df_ablation.to_dict(),
    'detailed_results': config_results,
    'example_queries': example_queries
}

with open('evaluation_results.json', 'w') as f:
    json.dump(all_eval_results, f, indent=2)

print("\n✅ Evaluation complete! Results saved to 'evaluation_results.json'")
print("✅ Ablation study table created!")


EXAMPLE OUTPUTS FOR REPORT (3 Queries)

QUERY: When did Pakistan gain independence from British rule?

📝 ANSWER:
August 1947

📚 RETRIEVED CONTEXT (first 300 chars of top chunk):
Pakistan became independent from the United Kingdom in 1947, but remained a British Dominion, like Canada and Australia, until 1956. Under Section 8 of the Indian Independence Act, 1947, the Government of India Act 1935 - with certain adaptations - served as the working constitution of Pakistan; sti...
⚠ Rate limit, waiting 3s... (attempt 1/3)

📊 EVALUATION SCORES:
  Faithfulness: 1.00
  Relevancy: 0.63

🔍 FAITHFULNESS DETAIL (first 500 chars):
CLAIMS:
August 1947 is a date mentioned in the context

VERIFICATION:
August 1947: SUPPORTED - The context mentions "On 15 August 1947, the new Dominion of Pakistan" which directly supports the claim.

SCORE: 1.00

SUMMARY: The answer "August 1947" is a faithful claim as it is directly supported by the context.

💡 GENERATED QUESTIONS FOR RELEVANCY:
  1. When did India g